# Experiment 5: Data Validation and Reproducible ML Pipelines

### AIM
Design and validate schema-aware data preprocessing workflows by identifying datatype mismatches, schema inconsistencies, missing attributes, invalid categorical values, and corrupted records. Build reusable and idempotent preprocessing pipelines and validate transformed features, preprocessing outputs, and dataset consistency across repeated executions.

---
### Workflow Architecture:
1. **Pandera Schema Specification**: Formal rules for news headlines, categorical validity (42 categories), ISO dates, and HTTP URLs.
2. **Baseline Dataset Validation**: Validates clean raw incoming data against the schema.
3. **Deliberate Fault Injection Test**: Injects 6 real-world corruptions:
   - Datatype mismatch (numeric headline)
   - Missing required attribute (NaN category)
   - Unrecognized categorical label (`"FAKE_NEWS_UNAUTHORIZED_CATEGORY"`)
   - Invalid date format (`"2026-99-99"`)
   - Corrupted URL scheme (`"ftp://..."`)
   - Schema inconsistency (unauthorized extra column)
4. **Audit Reporting**: Exports detailed failure diagnostics to `artifacts/schema_validation_failures.csv`.
5. **Reusable & Idempotent Preprocessing Pipeline**: Custom `TextCleanerTransformer` + `TfidfVectorizer` serialized to `models/preprocessor_pipeline.pkl`.
6. **Feature Matrix Integrity Audit**: Verifies 0 NaNs, shape consistency, label alignment, and value ranges.
7. **Idempotence & Determinism Proof**: Validates that $clean(clean(x)) = clean(x)$ and repeated pipeline transformations produce zero drift.


In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f"Working Directory set to: {PROJECT_ROOT}")

## 1. Define Production Pandera Schema

In [ ]:
from src.schema_validator import get_news_category_schema

schema = get_news_category_schema(strict=True)
print("Pandera Schema Columns and Constraints:")
for col_name, col_spec in schema.columns.items():
    print(f"  Column: '{col_name}' | Type: {col_spec.dtype} | Nullable: {col_spec.nullable}")

## 2. Validate Raw Clean Baseline Data

In [ ]:
from src.data_loader import load_raw_dataframe
from src.schema_validator import validate_schema

df = load_raw_dataframe()
is_valid, _ = validate_schema(df.head(1000), strict=False)
print(f"Baseline Data Valid: {is_valid}")

## 3. Deliberate Fault Injection & Corruptions Trap Test

In [ ]:
import pandas as pd
from src.schema_validator import inject_synthetic_corruptions

# Inject 6 real-world corruptions into test sample
corrupted_df = inject_synthetic_corruptions(df, num_corruptions=6)
is_valid_corrupt, failures = validate_schema(corrupted_df, strict=True)

print(f"Corrupted Data Passed: {is_valid_corrupt} (Expected: False)")
print("
Caught Violations Table:")
display(failures[['column', 'check', 'failure_case', 'index']])

## 4. Execute Reusable & Idempotent Preprocessing Pipeline

In [ ]:
from src.data_loader import prepare_train_test_split
from src.preprocessor_pipeline import run_preprocessor_pipeline

X_train_raw, X_test_raw, y_train, y_test = prepare_train_test_split(df)
pipeline, X_tr_mat, X_te_mat = run_preprocessor_pipeline(
    X_train_raw, X_test_raw, y_train, y_test, max_features=25000, ngram_range=(1, 2)
)

## 5. Transformed Output Feature Quality Audit

In [ ]:
from src.validate_exp5 import validate_transformed_features

audit_results = validate_transformed_features()
for k, v in audit_results.items():
    print(f"  {k}: {v}")

## 6. Mathematical Verification of Pipeline Idempotence

In [ ]:
from src.validate_exp5 import verify_pipeline_idempotence

idempotence_ok = verify_pipeline_idempotence(X_train_raw)
print(f"Idempotence and Zero-Drift Verified: {idempotence_ok}")

## 7. Display Preprocessing Summary Audit Report

In [ ]:
import json

with open("artifacts/preprocessing_summary_report.json") as f:
    report = json.load(f)
print(json.dumps(report, indent=4))